In [1]:
import pandas as pd
import numpy as np
import utils
from thefuzz import process, fuzz

df = pd.read_csv("data/league-fixture/csv/bundesliga_2023-2024_fixture_data.csv")
position_df = pd.read_csv("data/omer/Bundesliga_2015_to_2025.csv")

In [18]:
# Do it for on csv file. Later iterate it.
team_rankings = dict(zip(position_df.loc[position_df["Season"] == "23/24", "Team"], position_df.loc[position_df["Season"] == "23/24", "Position"]))

df = df[["Wk", "Home", "Away", "Score"]]
df = df.astype({"Wk": int})

# Add the goals scored by home and away teams.
df["HG"] = df.Score.str[:1].astype(int)
df["AG"] = df.Score.str[-1:].astype(int)

# Add the rankings of home and away teams at the end of the season
df["HP"] = df["Home"].apply(lambda desc: utils.find_best_match(desc, team_rankings))
df["AP"] = df["Away"].apply(lambda desc: utils.find_best_match(desc, team_rankings))

# Add the result of the match -1: Home win, : Draw, 1: Away win
df["Result"] = np.select(
    [
        df["HG"] > df["AG"],
        df["HG"] == df["AG"],
        df["HG"] < df["AG"]
    ],
    [-1, 0, 1] 
)

In [19]:
df.head(18)

,Wk,Home,Away,Score,HG,AG,HP,AP,Result
0,1,Werder Bremen,Bayern Munich,0–4,0,4,9,3,1
1,1,Wolfsburg,Heidenheim,2–0,2,0,12,8,-1
2,1,Stuttgart,Bochum,5–0,5,0,2,16,-1
3,1,Augsburg,Gladbach,4–4,4,4,11,14,0
4,1,Hoffenheim,Freiburg,1–2,1,2,7,10,1
5,1,Leverkusen,RB Leipzig,3–2,3,2,1,4,-1
6,1,Dortmund,Köln,1–0,1,0,5,17,-1
7,1,Union Berlin,Mainz 05,4–1,4,1,15,13,-1
8,1,Eint Frankfurt,Darmstadt 98,1–0,1,0,6,18,-1
9,2,RB Leipzig,Stuttgart,5–1,5,1,4,2,-1


In [14]:
np.where( df["HG"].values > df["AG"].values, -1, 0 )

array([ 0, -1, -1,  0,  0, -1, -1, -1, -1, -1, -1,  0,  0,  0,  0,  0,  0,
       -1,  0,  0, -1, -1, -1, -1,  0,  0,  0,  0, -1, -1,  0,  0,  0,  0,
       -1,  0, -1,  0, -1, -1,  0, -1, -1, -1,  0,  0,  0, -1,  0, -1,  0,
        0, -1, -1,  0,  0, -1,  0, -1,  0, -1, -1, -1, -1,  0, -1,  0,  0,
        0,  0, -1,  0,  0,  0, -1, -1, -1, -1, -1,  0, -1,  0,  0, -1,  0,
        0,  0,  0,  0, -1, -1, -1,  0,  0, -1,  0, -1,  0, -1,  0, -1,  0,
        0, -1,  0,  0,  0,  0,  0, -1, -1, -1, -1,  0,  0, -1, -1, -1, -1,
        0, -1, -1,  0,  0,  0,  0,  0,  0,  0, -1, -1, -1, -1, -1,  0,  0,
        0, -1, -1, -1, -1, -1,  0, -1,  0,  0,  0,  0,  0,  0,  0, -1,  0,
       -1, -1,  0,  0,  0,  0,  0, -1, -1, -1,  0,  0,  0, -1,  0, -1, -1,
        0, -1,  0,  0,  0,  0, -1,  0, -1,  0, -1,  0,  0,  0, -1,  0, -1,
       -1,  0,  0, -1,  0,  0,  0,  0, -1,  0, -1, -1,  0,  0, -1,  0, -1,
        0,  0, -1,  0,  0,  0,  0,  0,  0,  0,  0, -1, -1, -1, -1,  0, -1,
        0,  0, -1, -1,  0

In [15]:
df["HG"].values

array([0, 2, 5, 4, 1, 3, 1, 4, 1, 5, 1, 1, 2, 1, 1, 0, 1, 3, 2, 2, 5, 3,
       4, 5, 1, 1, 0, 2, 3, 2, 1, 2, 1, 1, 4, 3, 3, 0, 1, 2, 0, 7, 2, 4,
       0, 1, 0, 2, 1, 1, 0, 2, 4, 2, 2, 1, 3, 0, 4, 2, 3, 3, 2, 1, 1, 2,
       0, 1, 1, 1, 3, 2, 2, 2, 2, 2, 3, 8, 6, 3, 2, 1, 3, 2, 1, 0, 2, 0,
       2, 2, 4, 2, 1, 0, 4, 1, 4, 2, 3, 0, 4, 1, 1, 2, 0, 1, 0, 1, 0, 2,
       2, 3, 2, 0, 1, 2, 3, 2, 3, 0, 3, 5, 2, 1, 0, 2, 0, 1, 0, 3, 3, 2,
       3, 3, 1, 3, 1, 2, 3, 4, 2, 3, 1, 3, 0, 1, 0, 1, 0, 0, 1, 3, 0, 1,
       3, 2, 1, 2, 0, 1, 1, 1, 5, 1, 1, 2, 3, 0, 1, 3, 0, 3, 1, 1, 0, 0,
       2, 2, 2, 1, 3, 2, 0, 1, 1, 1, 3, 3, 1, 0, 1, 1, 0, 1, 1, 2, 3, 3,
       2, 1, 1, 5, 2, 2, 2, 2, 2, 2, 1, 0, 1, 1, 0, 2, 0, 2, 2, 1, 8, 3,
       2, 1, 1, 3, 2, 1, 1, 2, 2, 1, 2, 0, 2, 3, 0, 0, 0, 0, 2, 0, 1, 3,
       2, 1, 3, 2, 4, 1, 0, 0, 3, 1, 2, 1, 4, 1, 3, 2, 3, 0, 5, 3, 1, 1,
       0, 4, 1, 2, 1, 1, 3, 1, 2, 4, 0, 2, 0, 1, 0, 1, 2, 3, 3, 5, 0, 3,
       1, 1, 0, 1, 1, 1, 3, 3, 0, 2, 0, 1, 2, 4, 2,

array([ 1, -1, -1,  0,  1, -1, -1, -1, -1, -1, -1,  0,  1,  1,  1,  1,  0,
       -1,  0,  0, -1, -1, -1, -1,  1,  0,  1,  0, -1, -1,  1,  1,  1,  0,
       -1,  0, -1,  1, -1, -1,  1, -1, -1, -1,  0,  1,  1, -1,  1, -1,  1,
        0, -1, -1,  0,  1, -1,  0, -1,  1, -1, -1, -1, -1,  1, -1,  1,  1,
        1,  1, -1,  1,  0,  1, -1, -1, -1, -1, -1,  0, -1,  1,  0, -1,  0,
        1,  1,  1,  0, -1, -1, -1,  0,  0, -1,  0, -1,  0, -1,  1, -1,  0,
        0, -1,  1,  1,  0,  0,  1, -1, -1, -1, -1,  1,  0, -1, -1, -1, -1,
        1, -1, -1,  1,  0,  0,  0,  1,  0,  1, -1, -1, -1, -1, -1,  0,  0,
        0, -1, -1, -1, -1, -1,  1, -1,  0,  0,  1,  0,  1,  1,  0, -1,  1,
       -1, -1,  0,  0,  1,  1,  1, -1, -1, -1,  0,  0,  1, -1,  0, -1, -1,
        0, -1,  0,  1,  1,  1, -1,  0, -1,  0, -1,  0,  0,  0, -1,  1, -1,
       -1,  0,  1, -1,  0,  1,  1,  1, -1,  0, -1, -1,  0,  0, -1,  0, -1,
        0,  1, -1,  0,  1,  1,  1,  0,  1,  1,  1, -1, -1, -1, -1,  0, -1,
        1,  1, -1, -1,  1